<a href="https://colab.research.google.com/github/zizimostafa/MNIST-DL-Project/blob/main/Final_deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# Load Dataset

In [33]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# CNN Model

In [34]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

# Training + Testing Function

In [35]:
def train_and_evaluate(optimizer_name):

    model = SimpleCNN().to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=0.001)

    else:
        optimizer = optim.SGD(model.parameters(), lr=0.01)

    epochs = 5

    for epoch in range(epochs):

        model.train()

        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        acc = 100 * correct / total

        print(f"{optimizer_name} Epoch {epoch+1} | Loss: {total_loss:.4f} | Acc: {acc:.2f}%")

    # TESTING
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    test_acc = 100 * correct / total

    print(f"\n{optimizer_name} TEST ACCURACY: {test_acc:.2f}%\n")

    return test_acc

In [36]:
adam_acc = train_and_evaluate("Adam")
sgd_acc = train_and_evaluate("SGD")

Adam Epoch 1 | Loss: 169.9445 | Acc: 94.52%
Adam Epoch 2 | Loss: 45.8717 | Acc: 98.50%
Adam Epoch 3 | Loss: 31.6840 | Acc: 98.93%
Adam Epoch 4 | Loss: 23.5967 | Acc: 99.20%
Adam Epoch 5 | Loss: 17.7538 | Acc: 99.40%

Adam TEST ACCURACY: 99.15%

SGD Epoch 1 | Loss: 942.4378 | Acc: 72.54%
SGD Epoch 2 | Loss: 258.2948 | Acc: 91.61%
SGD Epoch 3 | Loss: 178.8097 | Acc: 94.28%
SGD Epoch 4 | Loss: 134.9530 | Acc: 95.60%
SGD Epoch 5 | Loss: 108.6073 | Acc: 96.48%

SGD TEST ACCURACY: 96.99%




# Comparison Table

In [37]:
print("===== FINAL COMPARISON =====")
print(f"Adam Accuracy: {adam_acc:.2f}%")
print(f"SGD Accuracy: {sgd_acc:.2f}%")

===== FINAL COMPARISON =====
Adam Accuracy: 99.15%
SGD Accuracy: 96.99%
